In [1]:
import pandas as pd
import numpy as np

In [2]:
caminho_csv = r"C:\Users\mateus_s_cavalheiro\Documents\GitHub\VulcanTech-Smart-Metals-S.A\data\raw\dados_sensores_vulcantech.csv"

# Leitura do dataset bruto
dataset = pd.read_csv(caminho_csv, sep=",")

# Padronização dos nomes das 11 colunas na ordem exata da especificação
dataset.columns = [
    "timestamp",
    "id_celula",
    "liga_metalica",
    "temperatura_fuso_c",
    "vibracao_triaxial_rms_mms",
    "torque_fuso_nm",
    "vazao_refrigerante_lmin",
    "corrente_estator_amp",
    "pressao_hidraulica_bar",
    "nivel_ruido_db",
    "tipo_falha"
]

print(f"Dimensões do dataset: {dataset.shape}")
dataset.head()

Dimensões do dataset: (10000, 11)


,timestamp,id_celula,liga_metalica,temperatura_fuso_c,vibracao_triaxial_rms_mms,torque_fuso_nm,vazao_refrigerante_lmin,corrente_estator_amp,pressao_hidraulica_bar,nivel_ruido_db,tipo_falha
0,2026-10-01 00:00:00,CEL-202,Aco_Carbono_316L,56.04,NaN,95.68,26.31,23.73,168.79,53.96,0
1,2026-10-01 00:05:00,CEL-203,Aco_Carbono_316L,50.76,1.93,118.89,29.97,28.31,144.27,53.43,0
2,2026-10-01 00:10:00,CEL-203,Aco_Carbono_316L,54.95,2.09,96.19,29.16,36.11,159.42,54.23,0
3,2026-10-01 00:15:00,CEL-201,Aco_Carbono_316L,58.18,1.90,101.89,29.31,27.77,162.28,52.93,0
4,2026-10-01 00:20:00,CEL-202,Liga_Titanio_Ti6Al4V,65.81,1.82,132.07,27.21,39.15,147.51,73.67,0


In [3]:
# Conversão de timestamp para Datetime
dataset['timestamp'] = pd.to_datetime(dataset['timestamp'])

# Ordenação temporal por equipamento
dataset = dataset.sort_values(by=['id_celula', 'timestamp']).reset_index(drop=True)
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   timestamp                  10000 non-null  datetime64[us]
 1   id_celula                  10000 non-null  str           
 2   liga_metalica              10000 non-null  str           
 3   temperatura_fuso_c         9750 non-null   float64       
 4   vibracao_triaxial_rms_mms  9750 non-null   float64       
 5   torque_fuso_nm             10000 non-null  float64       
 6   vazao_refrigerante_lmin    9800 non-null   float64       
 7   corrente_estator_amp       10000 non-null  float64       
 8   pressao_hidraulica_bar     10000 non-null  float64       
 9   nivel_ruido_db             10000 non-null  float64       
 10  tipo_falha                 10000 non-null  int64         
dtypes: datetime64[us](1), float64(7), int64(1), str(2)
memory usage: 859.5 KB


In [4]:
colunas_sensores = [
    'temperatura_fuso_c',
    'vibracao_triaxial_rms_mms',
    'torque_fuso_nm',
    'vazao_refrigerante_lmin',
    'corrente_estator_amp',
    'pressao_hidraulica_bar',
    'nivel_ruido_db'
]

# 1. Substituição de leituras fora dos limites operacionais físicos por NaN
dataset.loc[(dataset['temperatura_fuso_c'] >= 900) | (dataset['temperatura_fuso_c'] < 0), 'temperatura_fuso_c'] = np.nan
dataset.loc[dataset['vibracao_triaxial_rms_mms'] < 0, 'vibracao_triaxial_rms_mms'] = np.nan
dataset.loc[dataset['pressao_hidraulica_bar'] < 0, 'pressao_hidraulica_bar'] = np.nan
dataset.loc[dataset['vazao_refrigerante_lmin'] < 0, 'vazao_refrigerante_lmin'] = np.nan

# 2. Z-Score condicional por liga metálica para expurgar picos espúrios residuais (|Z| > 4)
for col in colunas_sensores:
    z_scores = dataset.groupby('liga_metalica')[col].transform(
        lambda x: (x - x.mean()) / x.std()
    )
    dataset.loc[z_scores.abs() > 4.0, col] = np.nan

In [5]:
# Quantificação da taxa de NaNs (%) agrupada por célula e liga metálica
taxa_nans = (
    dataset.groupby(['id_celula', 'liga_metalica'])[colunas_sensores]
    .apply(lambda g: (g.isna().sum() / len(g)) * 100)
    .round(2)
)
print("Taxa de NaNs (%) por Célula e Liga Metálica:")
display(taxa_nans)

# Interpolação temporal condicional por equipamento (id_celula)
dataset[colunas_sensores] = (
    dataset.groupby('id_celula')[colunas_sensores]
    .transform(lambda group: group.interpolate(method='linear').bfill().ffill())
)

print("\nValores nulos restantes após interpolação:")
print(dataset[colunas_sensores].isna().sum())

Taxa de NaNs (%) por Célula e Liga Metálica:


temperatura_fuso_c  vibracao_triaxial_rms_mms  \
id_celula liga_metalica                                                         
CEL-201   Aco_Carbono_316L                    3.98                       4.39   
          Aluminio_7075                       5.73                       3.18   
          Liga_Titanio_Ti6Al4V                3.29                       4.31   
CEL-202   Aco_Carbono_316L                    5.29                       4.05   
          Aluminio_7075                       3.18                       4.33   
          Liga_Titanio_Ti6Al4V                3.72                       3.91   
CEL-203   Aco_Carbono_316L                    4.21                       4.53   
          Aluminio_7075                       3.10                       3.95   
          Liga_Titanio_Ti6Al4V                5.03                       4.43   
CEL-204   Aco_Carbono_316L                    4.17                       3.51   
          Aluminio_7075                       3.57                       4.89   
          Liga_Titanio_Ti6Al4V                2.83                       3.24   

                                torque_fuso_nm  vazao_refrigerante_lmin  \
id_celula liga_metalica                                                   
CEL-201   Aco_Carbono_316L                0.41                     4.23   
          Aluminio_7075                   0.51                     4.84   
          Liga_Titanio_Ti6Al4V            1.23                     3.90   
CEL-202   Aco_Carbono_316L                0.70                     3.89   
          Aluminio_7075                   0.38                     4.96   
          Liga_Titanio_Ti6Al4V            0.59                     3.72   
CEL-203   Aco_Carbono_316L                0.89                     3.24   
          Aluminio_7075                   0.00                     4.37   
          Liga_Titanio_Ti6Al4V            0.60                     2.01   
CEL-204   Aco_Carbono_316L                0.82                     5.07   
          Aluminio_7075                   0.79                     5.56   
          Liga_Titanio_Ti6Al4V            1.01                     2.02   

                                corrente_estator_amp  pressao_hidraulica_bar  \
id_celula liga_metalica                                                        
CEL-201   Aco_Carbono_316L                      0.49                    2.20   
          Aluminio_7075                         0.51                    2.55   
          Liga_Titanio_Ti6Al4V                  1.03                    3.08   
CEL-202   Aco_Carbono_316L                      0.16                    1.79   
          Aluminio_7075                         0.89                    2.16   
          Liga_Titanio_Ti6Al4V                  0.20                    1.37   
CEL-203   Aco_Carbono_316L                      0.24                    2.02   
          Aluminio_7075                         0.00                    1.13   
          Liga_Titanio_Ti6Al4V                  0.80                    3.02   
CEL-204   Aco_Carbono_316L                      0.41                    1.31   
          Aluminio_7075                         0.40                    2.12   
          Liga_Titanio_Ti6Al4V                  0.81                    0.40   

                                nivel_ruido_db  
id_celula liga_metalica                         
CEL-201   Aco_Carbono_316L                0.65  
          Aluminio_7075                   0.64  
          Liga_Titanio_Ti6Al4V            0.62  
CEL-202   Aco_Carbono_316L                0.31  
          Aluminio_7075                   0.89  
          Liga_Titanio_Ti6Al4V            0.59  
CEL-203   Aco_Carbono_316L                0.57  
          Aluminio_7075                   0.85  
          Liga_Titanio_Ti6Al4V            1.21  
CEL-204   Aco_Carbono_316L                0.41  
          Aluminio_7075                   0.53  
          Liga_Titanio_Ti6Al4V            0.81


Valores nulos restantes após interpolação:
temperatura_fuso_c           0
vibracao_triaxial_rms_mms    0
torque_fuso_nm               0
vazao_refrigerante_lmin      0
corrente_estator_amp         0
pressao_hidraulica_bar       0
nivel_ruido_db               0
dtype: int64


In [6]:
# 1. Índice de Esforço Mecânico (IEM)
dataset['iem'] = (
    (dataset['torque_fuso_nm'] * dataset['vibracao_triaxial_rms_mms']) / 
    (dataset['vazao_refrigerante_lmin'] + 1)
)

# 2. Features dinâmicas temporais (janelas de 30 min e 2 horas)
features_dinamicas = ['temperatura_fuso_c', 'vibracao_triaxial_rms_mms', 'torque_fuso_nm', 'iem']

for col in features_dinamicas:
    dataset[f'{col}_rolling_mean_30min'] = dataset.groupby('id_celula')[col].transform(
        lambda x: x.rolling(window=6, min_periods=1).mean()
    )
    dataset[f'{col}_rolling_std_30min'] = dataset.groupby('id_celula')[col].transform(
        lambda x: x.rolling(window=6, min_periods=1).std().fillna(0)
    )
    dataset[f'{col}_rolling_mean_2h'] = dataset.groupby('id_celula')[col].transform(
        lambda x: x.rolling(window=24, min_periods=1).mean()
    )
    dataset[f'{col}_rolling_std_2h'] = dataset.groupby('id_celula')[col].transform(
        lambda x: x.rolling(window=24, min_periods=1).std().fillna(0)
    )

# 3. Normalização das variáveis de processo pela média histórica da liga metálica
for col in colunas_sensores:
    dataset[f'{col}_norm_liga'] = dataset[col] / dataset.groupby('liga_metalica')[col].transform('mean')

dataset.head()

,timestamp,id_celula,liga_metalica,temperatura_fuso_c,vibracao_triaxial_rms_mms,torque_fuso_nm,vazao_refrigerante_lmin,corrente_estator_amp,pressao_hidraulica_bar,nivel_ruido_db,...,iem_rolling_std_30min,iem_rolling_mean_2h,iem_rolling_std_2h,temperatura_fuso_c_norm_liga,vibracao_triaxial_rms_mms_norm_liga,torque_fuso_nm_norm_liga,vazao_refrigerante_lmin_norm_liga,corrente_estator_amp_norm_liga,pressao_hidraulica_bar_norm_liga,nivel_ruido_db_norm_liga
0,2026-10-01 00:15:00,CEL-201,Aco_Carbono_316L,58.18,1.90,101.89,29.31,27.77,162.28,52.93,...,0.000000,6.387034,0.000000,1.066029,0.877499,0.872541,1.084999,0.925210,1.029166,0.913939
1,2026-10-01 00:35:00,CEL-201,Aco_Carbono_316L,45.76,1.94,102.71,32.53,27.18,169.21,53.80,...,0.314220,6.164847,0.314220,0.838458,0.895973,0.879563,1.204198,0.905553,1.073116,0.928961
2,2026-10-01 00:40:00,CEL-201,Aco_Carbono_316L,53.65,1.42,95.06,25.67,31.14,152.83,53.54,...,0.674757,5.797002,0.674757,0.983026,0.655815,0.814052,0.950254,1.037488,0.969235,0.924472
3,2026-10-01 01:00:00,CEL-201,Aluminio_7075,70.92,2.54,103.13,27.39,31.02,166.93,45.08,...,1.801247,6.654464,1.801247,1.624843,1.491518,1.108559,0.996657,1.305040,1.054574,0.980388
4,2026-10-01 01:05:00,CEL-201,Aluminio_7075,61.19,1.40,113.46,29.11,20.19,162.58,48.46,...,1.677409,6.378662,1.677409,1.401920,0.822096,1.219597,1.059244,0.849412,1.027093,1.053895


In [7]:
import os

print(f"Dimensões finais com novas features: {dataset.shape}")
print("\nDistribuição das classes da variável alvo (tipo_falha):")
print(dataset['tipo_falha'].value_counts(dropna=False))

# Exportação do dataset tratado
caminho_saida = r"C:\Users\mateus_s_cavalheiro\Documents\GitHub\VulcanTech-Smart-Metals-S.A\data\processed\dados_sensores_tratados.csv"
os.makedirs(os.path.dirname(caminho_saida), exist_ok=True)
dataset.to_csv(caminho_saida, index=False)
print(f"\nArquivo salvo com sucesso em: {caminho_saida}")

Dimensões finais com novas features: (10000, 35)

Distribuição das classes da variável alvo (tipo_falha):
tipo_falha
0    8557
2     509
1     470
3     464
Name: count, dtype: int64

Arquivo salvo com sucesso em: C:\Users\mateus_s_cavalheiro\Documents\GitHub\VulcanTech-Smart-Metals-S.A\data\processed\dados_sensores_tratados.csv
